### Installing and importing necessary libraries and setup

In [ ]:
!pip install open_clip_torch seaborn tqdm -q

In [ ]:
import os
import time
import csv

import torch
import open_clip
from PIL import Image
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

In [ ]:
# Setting device to GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device}")

### Setting up Dataset

In [ ]:
# UNZIPPING FILES

!unzip -q "/content/final_dataset.zip" -d "/content/"

In [ ]:
DATASET_PATH = "/content/final_dataset"

In [ ]:
CATEGORIES = sorted([
    f for f in os.listdir(DATASET_PATH)
    if os.path.isdir(os.path.join(DATASET_PATH, f))
])

print("Categories Found:", CATEGORIES)

In [ ]:
all_image_paths = []
all_true_labels = []

for category in CATEGORIES:
  folder = os.path.join(DATASET_PATH, category)
  images = [
      f for f in os.listdir(folder)
      if f.lower().endswith(".jpg")
  ]
  for img in images:
    all_image_paths.append(os.path.join(folder, img))
    all_true_labels.append(category)
  print(f"{category}: {len(images)} images")

print(f"Total Images: {len(all_image_paths)}")

### Writing prompts
Each category should be explained in an egnlish description

In [ ]:
PROMPTS = {
    "animal": {
        "primary": "a photograph of an animal such as a dog, cat, bird, or wildlife, where the animal is the main subject of the image",
        "ensemble": [
            "a photograph of an animal such as a dog, cat, bird, or wildlife, where the animal is the main subject of the image",
            "a nature or pet photograph featuring an animal as the central subject, without any text overlays or graphic design elements",
            "a close-up or full-body photograph of a real animal in a natural setting, home, or studio",
            "a wildlife or domestic animal photograph showing a single creature or group of animals",
        ]
    },

    "codesnippet": {
        "primary": "a code snippet or block of programming code displayed with syntax highlighting, typically on a dark background in a code editor or terminal",
        "ensemble": [
            "a code snippet or block of programming code displayed with syntax highlighting, typically on a dark background in a code editor or terminal",
            "a screenshot or styled image of source code with monospace font, line numbers, and coloured syntax highlighting",
            "programming code shown in an IDE, terminal window, or as a styled code card shared on social media",
            "a block of computer code with keywords highlighted in different colours, shown against a dark or light editor background",
        ]
    },

    "documents": {
        "primary": "a scanned or photographed text document such as a letter, contract, form, or office file containing paragraphs of text with no itemized pricing or receipt format",
        "ensemble": [
            "a scanned or photographed text document such as a letter, contract, form, or office file containing paragraphs of text with no itemized pricing or receipt format",
            "an office document or formal letter with blocks of paragraph text, headings, and no product pricing or transaction totals",
            "a printed or digital document such as a report, agreement, or form with continuous prose text and standard document formatting",
            "a text-heavy official document including applications, certificates, contracts, or letters, without any columns of prices or itemized lists",
        ]
    },

    "food": {
        "primary": "a close-up photograph of a prepared meal, dish, or food item presented on a plate, bowl, or surface, with food as the sole subject",
        "ensemble": [
            "a close-up photograph of a prepared meal, dish, or food item presented on a plate, bowl, or surface, with food as the sole subject",
            "a food photography shot of a dish, snack, dessert, or drink taken in a restaurant, kitchen, or studio setting",
            "an appetizing photograph of cooked or prepared food arranged on a plate or table, shot from above or at an angle",
            "a photograph showcasing a specific food item or full meal, emphasizing colour and texture, without people as the main focus",
        ]
    },

    "infographic": {
        "primary": "an infographic combining icons, charts, diagrams, and short text labels arranged in a structured visual layout to present information or data",
        "ensemble": [
            "an infographic combining icons, charts, diagrams, and short text labels arranged in a structured visual layout to present information or data",
            "a designed visual explainer with colourful icons, numbered sections, pie charts or bar graphs, and brief captions",
            "an information graphic or data visualization poster that uses illustrations and minimal text to communicate statistics or steps",
            "a visually designed educational or marketing graphic with a mix of icons, short bullet points, and structured sections on a coloured background",
        ]
    },

    "meme": {
        "primary": "an internet meme with bold white or black text caption overlaid directly on a photograph or illustration, typically humorous or satirical in nature",
        "ensemble": [
            "an internet meme with bold white or black text caption overlaid directly on a photograph or illustration, typically humorous or satirical in nature",
            "a viral internet meme image featuring a photograph or drawing with large impact font text written across the top and bottom",
            "a humorous meme image where text has been added on top of a photo or reaction image to create a joke or commentary",
            "a social media meme consisting of an image macro with superimposed bold text used for comedic or satirical effect",
            "a slightly distorted or out of proportion photo of people "
        ]
    },

    "real_people": {
        "primary": "a natural photograph of one or more real people such as a portrait, selfie, candid shot, or group photo, with no text overlaid on the image",
        "ensemble": [
            "a natural photograph of one or more real people such as a portrait, selfie, candid shot, or group photo, with no text overlaid on the image",
            "a candid or posed photograph of a real human being, showing their face or full body in a natural or studio environment",
            "a portrait or lifestyle photograph featuring a person or group of people as the main subject, without meme text or graphic design elements",
            "a real photograph of a human subject including selfies, professional headshots, street photography, or group photos without any caption text",
        ]
    },

    "receipt": {
        "primary": "a printed or photographed receipt, bill, or invoice showing a narrow columnar list of purchased items with individual prices and a total amount due at the bottom",
        "ensemble": [
            "a printed or photographed receipt, bill, or invoice showing a narrow columnar list of purchased items with individual prices and a total amount due at the bottom",
            "a transaction receipt from a shop, restaurant, or online order showing item names, quantities, unit prices, taxes, and a grand total",
            "a paper or digital receipt with a structured list of line items and monetary amounts, typically narrow in format with a store name at the top",
            "a purchase receipt or payment bill with itemized pricing columns, subtotal, tax, and total due, printed in small monospace font",
        ]
    },

    "wallpapers": {
        "primary": "a high resolution decorative image specifically composed for use as a desktop or phone wallpaper, featuring artistic lighting, wide framing, and a visually striking subject",
        "ensemble": [
            "a high resolution decorative image specifically composed for use as a desktop or phone wallpaper, featuring artistic lighting, wide framing, and a visually striking subject",
            "a professionally composed wide-format photograph or digital artwork designed to fill a computer or phone screen as a background image",
            "a visually polished landscape, abstract, or artistic photograph with widescreen aspect ratio and high production quality intended as a screen background",
            "a desktop wallpaper image with cinematic framing, vivid colours, and deliberate artistic composition, designed to be aesthetically pleasing at full screen size",
        ]
    },
}
# Sanity check - every category must have a prompt
missing = [c for c in CATEGORIES if c not in PROMPTS]
if missing:
    print(f"WARNING: no prompt written for: {missing}")
    print("Add them to PROMPTS above before continuing.")
else:
    print("All categories have prompts. Ready.")

### Definng what models to test
there are 9 models to test
1. 'RN50'
2.    'RN101',
3.    'RN50x4',
4.    'RN50x16',
5.    'RN50x64',
6.    'ViT-B/32',
7.    'ViT-B/16',
8.    'ViT-L/14',
9.    'ViT-L/14@336px'

In [ ]:
MODELS_TO_TEST = [
    ("ViT-B-32", "openai"),
    ("ViT-B-16", "openai"),
    ("ViT-L-14", "openai"),
    ("RN50", "openai"),
    ("RN101", "openai"),
    ("RN50x4", "openai"),
    ("RN50x16", "openai"),
    ("RN50x64", "openai"),
    # ("ViT-L-14@336px", "openai"),
]

print(f"Will benchmark {len(MODELS_TO_TEST)} models: ")
for name, weights in MODELS_TO_TEST:
  print(f" {name} ({weights})")

### Benchmark Function

In [ ]:
def benchmark_model(model_name, pretrained, image_paths, true_labels, categories, prompts, device):
  """
  Loading one CLIP model, run every image through it and returns the results.

  What's happening inside:
   1. Load the model and move it to GPU (or CPU if no GPU)
   2. Encode all category prompts into embeddings once - not once per image,
      once total. These don't change between images.
   3. For every image: encode it, compare to every prompt, pick the closest one
      as the prediction.
   4. Record whether it was right or wrong, and how confident it was
  """

  print(f"\n{'='*50}")
  print(f"Benchmarking: {model_name} ({pretrained})")
  print(f"{'='*50}")

  # Loading the model
  load_start = time.time()
  model, _, preprocess = open_clip.create_model_and_transforms(
      model_name, pretrained=pretrained
  )
  model = model.to(device).eval()
  tokenizer = open_clip.get_tokenizer(model_name)
  load_time = time.time() - load_start
  print(f"Model loaded in {load_time: 1f}s")

  # Encoding category prompt once
  prompt_text = [prompts[c]['primary'] for c in categories] # Fixed: Access 'primary' prompt
  with torch.no_grad():
    text_tokens = tokenizer(prompt_text).to(device)
    text_features = model.encode_text(text_tokens)
    text_features /= text_features.norm(dim=-1, keepdim=True)

  # Run every image through the model
  predicted_labels = []
  confidence_scores = []
  inference_times = []
  errors = []

  for image_path in tqdm(image_paths, desc = f"{model_name}"):
    try:
      t_start = time.time()
      image = preprocess(Image.open(image_path).convert("RGB")).unsqueeze(0).to(device)

      with torch.no_grad():
        image_features = model.encode_image(image)
        image_features /= image_features.norm(dim=-1, keepdim=True)
        probs = (image_features @ text_features.T).softmax(dim=-1)

      t_end = time.time()

      probs_np = probs[0].cpu().numpy()
      best_idx = probs_np.argmax()

      predicted_labels.append(categories[best_idx])
      confidence_scores.append(float(probs_np[best_idx]))
      inference_times.append((t_end - t_start)*1000)

    except Exception as e:
      print(f"\n Skipped {os.path.basename(image_path)} : {e}")
      errors.append(image_path)
      predicted_labels.append(None)
      confidence_scores.append(None)
      inference_times.append(None)


  # These calculations should be done after the loop finishes
  correct_confs = [
      conf for pred, true, conf in zip(predicted_labels, true_labels, confidence_scores)
      if pred == true
  ]

  wrong_confs = [
      conf for pred, true, conf in zip(predicted_labels, true_labels, confidence_scores)
      if pred != true and pred != "ERROR"
  ]

  results = {
      "model" : f"{model_name} ({pretrained})",
      "predicted_labels": predicted_labels,
      "confidence_scores": confidence_scores,
      "inference_times_ms": inference_times,
      "avg_inference_ms": np.mean([t for t in inference_times if t is not None and t > 0]),
      "avg_conf_correct": np.mean(correct_confs) if correct_confs else 0,
      "avg_conf_wrong": np.mean(wrong_confs) if wrong_confs else 0,
      "errors": errors,
  }


  print(f"Done. Avg inferences: {results['avg_inference_ms']:.1f}ms/img")

  # Freeing GPU memory before loading the next model
  del model
  torch.cuda.empty_cache()

  return results

In [ ]:
all_results = []

for model_name, pretrained in MODELS_TO_TEST:
  result = benchmark_model(
      model_name = model_name,
      pretrained = pretrained,
      image_paths = all_image_paths,
      true_labels = all_true_labels,
      categories = CATEGORIES,
      prompts = PROMPTS,
      device = device
  )

  all_results.append(result)

print("\nAll models done.")

### Printing the metrics

In [ ]:
summary_rows = []

for result in all_results:
    model_label    = result["model"]
    predicted      = result["predicted_labels"]
    times          = result["inference_times_ms"]

    # Filter out any images that errored
    valid = [(p, t, c) for p, t, c in
             zip(predicted, all_true_labels, result["confidence_scores"])
             if p != "ERROR"]
    valid_pred  = [v[0] for v in valid]
    valid_true  = [v[1] for v in valid]

    correct = sum(p == t for p, t in zip(valid_pred, valid_true))
    total   = len(valid_true)
    overall_accuracy = correct / total * 100

    print(f"\n{'='*60}")
    print(f"MODEL: {model_label}")
    print(f"{'='*60}")
    print(f"Overall accuracy:          {overall_accuracy:.1f}%  ({correct}/{total})")
    print(f"Avg inference time:        {result['avg_inference_ms']:.1f} ms/image")
    print(f"Avg confidence (correct):  {result['avg_conf_correct']:.1%}")
    print(f"Avg confidence (wrong):    {result['avg_conf_wrong']:.1%}")

    if result["errors"]:
        print(f"Images skipped (errors):   {len(result['errors'])}")

    # Per-category breakdown
    print(f"\nPer-category accuracy:")
    print(classification_report(valid_true, valid_pred, target_names=CATEGORIES, digits=3))

    # Row for the summary table
    summary_rows.append({
        "Model":               model_label,
        "Overall Accuracy %":  round(overall_accuracy, 2),
        "Avg ms/image":        round(result["avg_inference_ms"], 1),
        "Conf when correct":   round(result["avg_conf_correct"], 3),
        "Conf when wrong":     round(result["avg_conf_wrong"], 3),
    })

# Summary table across all models
print("\n\n" + "="*60)
print("SUMMARY — all models side by side")
print("="*60)
summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

### Confusion Matrix


In [ ]:
# One confusion matrix per model, side by side.
# Each row = true label, each column = what the model predicted.
# A perfect model has all the numbers on the diagonal and zeros everywhere else.
# Wherever you see a big number off the diagonal, that's a pair of categories
# the model keeps confusing with each other.

fig, axes = plt.subplots(1, len(all_results),
                         figsize=(7 * len(all_results), 6))

if len(all_results) == 1:
    axes = [axes]

for ax, result in zip(axes, all_results):
    valid = [(p, t) for p, t in zip(result["predicted_labels"], all_true_labels)
             if p != "ERROR"]
    valid_pred = [v[0] for v in valid]
    valid_true = [v[1] for v in valid]

    cm = confusion_matrix(valid_true, valid_pred, labels=CATEGORIES)

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CATEGORIES,
        yticklabels=CATEGORIES,
        ax=ax,
    )
    ax.set_title(result["model"], fontsize=11)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.savefig("/content/confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: /content/confusion_matrices.png")

### Saving everything to CSV

In [ ]:
csv_path = "/content/initial_benchmark_results.csv"

with open(csv_path, "w", newline="") as f:
  writer = csv.writer(f)
  writer.writerow([
      "model", "image_path","filename", "true_label","predicted_label", "correct",
      "confidence_score", "inference_ms"
  ])

  for result in all_results:
    for img_path, true, pred, conf, ms in zip(
        all_image_paths,
        all_true_labels,
        result["predicted_labels"],
        result["confidence_scores"],
        result["inference_times_ms"],
    ):
      writer.writerow([
          result["model"],
          img_path,
          os.path.basename(img_path),
          true,
          pred,
          pred == true,
          round(conf, 4),
          round(ms,1),
      ])

  print(f"Saved raw results:  {csv_path}")

  # Copying filed to drive so they survive runtime disconnect
  import shutil
  shutil.copy(csv_path, "/content/drive/MyDrive/initial_benchmark_results.csv")
  shutil.copy("/content/confusion_matrices.png",
              "/content/drive/MyDrive/confusion_matrices.png")
  print("Content Copied to Drrive")


